# Quantum Emitters in Non-Hermitian Nanophotonic SSH Lattices
## Local Density of States and Edge-Enhanced Light–Matter Interaction

**Notebook:** 05_emitters_ldos_nanophotonics  

In this notebook, quantum emitters are introduced as probes of the photonic SSH
lattice. The focus is on how the spectral and spatial properties of the lattice
modes influence light–matter interaction at specific sites.

The analysis is based on the local density of states (LDOS), which provides a
direct measure of the photonic environment experienced by an emitter. By
comparing the LDOS at edge and bulk sites, we examine how topological edge modes
modify emission properties in a non-Hermitian nanophotonic system.

This notebook links the topological lattice models developed earlier to a
physically relevant observable from quantum optics, providing a concrete
connection between topology, nanophotonics, and emitter dynamics. The notebook
is fully self-contained and can be run independently.

## Scope and Physical Model

A quantum emitter is modeled as a two-level system weakly coupled to the
photonic SSH lattice. The analysis is restricted to the single-excitation
sector, where the emitter interacts with the photonic modes without inducing
nonlinear effects.

The photonic environment is described by an effective non-Hermitian
Hamiltonian, and the influence of the lattice on the emitter is characterized
through the local density of states (LDOS). In this regime, the LDOS directly
governs the spontaneous emission rate via Fermi’s golden rule.

Strong coupling effects, nonlinear dynamics, and full Maxwell–Bloch equations
are not considered, as they are not required for capturing the edge-enhanced
emission behavior studied here.

In [2]:
import numpy as np
import scipy.linalg as la
import matplotlib.pyplot as plt
import yaml
from pathlib import Path

In [3]:
def load_yaml(path):
    with open(path, "r") as f:
        return yaml.safe_load(f)

CONFIG_DIR = Path("..") / "config"

global_config = load_yaml(CONFIG_DIR / "global_config.yaml")
lattice_config = load_yaml(CONFIG_DIR / "lattice_params.yaml")
plotting_config = load_yaml(CONFIG_DIR / "plotting_params.yaml")

NOTEBOOK_KEY = "05_emitters_ldos_nanophotonics"
assert NOTEBOOK_KEY in plotting_config["notebook_figures"]

## Finite Photonic Hamiltonian

The photonic environment experienced by the emitter is modeled using the same
finite SSH lattice introduced earlier, including gain and loss terms. Working in
real space allows boundary effects to be captured explicitly, which is
essential for analyzing edge-localized modes and their influence on emitter
dynamics.

In [4]:
def build_nonhermitian_ssh_hamiltonian(
    num_cells,
    t1,
    t2,
    gamma_a,
    gamma_b
):
    # Building a real-space non-Hermitian SSH Hamiltonian with open boundaries.
    
    dim = 2 * num_cells
    H = np.zeros((dim, dim), dtype=np.complex128)

    for n in range(num_cells):
        a = 2 * n
        b = 2 * n + 1

        # On-site gain/loss
        H[a, a] = 1j * gamma_a
        H[b, b] = -1j * gamma_b

        # Intra-cell hopping
        H[a, b] = t1
        H[b, a] = t1

        # Inter-cell hopping
        if n < num_cells - 1:
            H[a + 2, b] = t2
            H[b, a + 2] = t2

    return H

## System Parameters